In [1]:
import sys
import radiomics
import SimpleITK as sitk
import numpy as np

print("Python:", sys.version)
print("PyRadiomics:", radiomics.__version__)
print("SimpleITK:", sitk.Version())
print("NumPy:", np.__version__)

Python: 3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]
PyRadiomics: v3.1.0
SimpleITK: SimpleITK Version: 2.5.6 (ITK 5.4)
Compiled: Jul 29 2026 20:26:40

NumPy: 1.26.4


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"F:\yingxiang2")

yaml_path = PROJECT_ROOT / "config" / "pyradiomics.yaml"

yaml_content = """\
imageType:
  Original: {}

featureClass:
  shape: []
  firstorder: []
  glcm: []
  glrlm: []
  glszm: []
  gldm: []
  ngtdm: []

setting:
  binWidth: 25
  resampledPixelSpacing: [1, 1, 1]
  interpolator: sitkBSpline
  normalize: false
"""

yaml_path.write_text(yaml_content, encoding="utf-8")

print("Saved to:")
print(yaml_path)
print("Exists:", yaml_path.exists())

Saved to:
F:\yingxiang2\config\pyradiomics.yaml
Exists: True


In [3]:
from radiomics import featureextractor

extractor = featureextractor.RadiomicsFeatureExtractor(
    str(yaml_path)
)

print("Enabled image types:")
print(extractor.enabledImagetypes)

print("\nEnabled feature classes:")
print(extractor.enabledFeatures)

print("\nSettings:")
print(extractor.settings)

Enabled image types:
{'Original': {}}

Enabled feature classes:
{'shape': [], 'firstorder': [], 'glcm': [], 'glrlm': [], 'glszm': [], 'gldm': [], 'ngtdm': []}

Settings:
{'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': [1, 1, 1], 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 25}


In [6]:
import pandas as pd
import pydicom
import numpy as np
import SimpleITK as sitk
from pathlib import Path

PROJECT_ROOT = Path(r"F:\yingxiang2")

qc_path = PROJECT_ROOT / "results" / "metrics" / "final_cohort_qc.csv"
qc_df = pd.read_csv(qc_path)

case_id = "R01-003"

row = qc_df.loc[qc_df["Case ID"] == case_id].iloc[0]

ct_folder = Path(row["Matched CT folder"])
seg_file = Path(row["SEG file"])

print("Case:", case_id)
print("CT folder:", ct_folder)
print("SEG file:", seg_file)

Case: R01-003
CT folder: F:\yingxiang2\data\raw\NSCLC Radiogenomics\R01-003\12-09-1991-NA-CT CHEST WO-02622\4.000000-THIN LUNG WINDOW-69868
SEG file: F:\yingxiang2\data\raw\NSCLC Radiogenomics\R01-003\12-09-1991-NA-CT CHEST WO-02622\1000.000000-3D Slicer segmentation result-96510\1-1.dcm


In [5]:
import pandas as pd
import pydicom
import numpy as np
import SimpleITK as sitk
from pathlib import Path

In [7]:
reader = sitk.ImageSeriesReader()

series_ids = reader.GetGDCMSeriesIDs(str(ct_folder))
print("Series IDs:", series_ids)

file_names = reader.GetGDCMSeriesFileNames(
    str(ct_folder),
    series_ids[0]
)

reader.SetFileNames(file_names)
ct_image = reader.Execute()

print("CT size:", ct_image.GetSize())
print("CT spacing:", ct_image.GetSpacing())
print("CT origin:", ct_image.GetOrigin())
print("CT direction:", ct_image.GetDirection())

Series IDs: ('1.3.6.1.4.1.14519.5.2.1.4334.1501.668993209202739148457939869868',)
CT size: (512, 512, 288)
CT spacing: (0.9375, 0.9375, 1.25)
CT origin: (-252.9, -246.2, -377.75)
CT direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)


In [8]:
seg_ds = pydicom.dcmread(seg_file)
seg_array = seg_ds.pixel_array

# 建立 CT SOPInstanceUID -> z-index 映射
ct_uid_to_index = {}

for idx, f in enumerate(file_names):
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    ct_uid_to_index[str(ds.SOPInstanceUID)] = idx

# 建立空的3D mask
mask_array = np.zeros(
    (
        ct_image.GetSize()[2],  # z
        ct_image.GetSize()[1],  # y
        ct_image.GetSize()[0],  # x
    ),
    dtype=np.uint8
)

matched_frames = 0

for frame_idx, frame_fg in enumerate(seg_ds.PerFrameFunctionalGroupsSequence):
    uid = None

    if hasattr(frame_fg, "DerivationImageSequence"):
        deriv = frame_fg.DerivationImageSequence[0]

        if hasattr(deriv, "SourceImageSequence"):
            src = deriv.SourceImageSequence[0]
            uid = str(getattr(src, "ReferencedSOPInstanceUID", ""))

    if uid in ct_uid_to_index:
        z_idx = ct_uid_to_index[uid]
        mask_array[z_idx] = (seg_array[frame_idx] > 0).astype(np.uint8)
        matched_frames += 1

print("SEG frames:", seg_array.shape[0])
print("Matched frames:", matched_frames)
print("Mask shape:", mask_array.shape)
print("Mask unique values:", np.unique(mask_array))
print("Mask nonzero voxels:", int(mask_array.sum()))

SEG frames: 288
Matched frames: 0
Mask shape: (288, 512, 512)
Mask unique values: [0]
Mask nonzero voxels: 0


In [9]:
# 看第一帧里到底有哪些字段
frame0 = seg_ds.PerFrameFunctionalGroupsSequence[0]

print("Frame 0 attributes:")
for elem in frame0:
    print(elem.keyword, "->", elem.value)

Frame 0 attributes:
FrameContentSequence -> [(0020, 9056) Stack ID                            SH: '1'
(0020, 9057) In-Stack Position Number            UL: 1
(0020, 9157) Dimension Index Values              UL: [1, 1]]
PlanePositionSequence -> [(0020, 0032) Image Position (Patient)            DS: [-252.900, -246.200, -377.750]]


In [10]:
print("\nSharedFunctionalGroupsSequence:")
if hasattr(seg_ds, "SharedFunctionalGroupsSequence"):
    shared = seg_ds.SharedFunctionalGroupsSequence[0]
    for elem in shared:
        print(elem.keyword, "->", elem.value)


SharedFunctionalGroupsSequence:
DerivationImageSequence -> [(0008, 2112)  Source Image Sequence  288 item(s) ---- 
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.177050854000720628302957773144
   (0008, 1160) Referenced Frame Number             IS: '1'
   (0040, a170)  Purpose of Reference Code Sequence  1 item(s) ---- 
      (0008, 0100) Code Value                          SH: '121322'
      (0008, 0102) Coding Scheme Designator            SH: 'DCM'
      (0008, 0104) Code Meaning                        LO: 'Source image for image processing operation'
      ---------
   ---------
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.288627475233742316637122617447
   (0008, 1160) Referenced Frame Number             IS: '1'
   (0040, a170)  Purpose of Reference Co

In [11]:
print("\nTop-level referenced sequences:")

for keyword in [
    "ReferencedSeriesSequence",
    "StudiesContainingOtherReferencedInstancesSequence",
    "ReferencedInstanceSequence"
]:
    if hasattr(seg_ds, keyword):
        print(keyword, "FOUND")
        print(getattr(seg_ds, keyword))


Top-level referenced sequences:
ReferencedSeriesSequence FOUND
[(0008, 114a)  Referenced Instance Sequence  288 item(s) ---- 
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.177050854000720628302957773144
   ---------
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.288627475233742316637122617447
   ---------
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.204010657945300393691783947304
   ---------
   (0008, 1150) Referenced SOP Class UID            UI: CT Image Storage
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.6.1.4.1.14519.5.2.1.4334.1501.290638244279219611329670460702
   ---------
   (0008, 1150) Referenced SOP Class UID     

In [12]:
# 检查 SEG 每一帧是否包含空间位置
frame_positions = []

for i, frame_fg in enumerate(seg_ds.PerFrameFunctionalGroupsSequence):
    position = None

    if hasattr(frame_fg, "PlanePositionSequence"):
        item = frame_fg.PlanePositionSequence[0]

        if hasattr(item, "ImagePositionPatient"):
            position = tuple(
                float(x) for x in item.ImagePositionPatient
            )

    frame_positions.append(position)

print("SEG frames:", len(frame_positions))
print(
    "Frames with ImagePositionPatient:",
    sum(p is not None for p in frame_positions)
)

print("\nFirst 5 SEG frame positions:")
for i, p in enumerate(frame_positions[:5]):
    print(i, p)

SEG frames: 288
Frames with ImagePositionPatient: 288

First 5 SEG frame positions:
0 (-252.9, -246.2, -377.75)
1 (-252.9, -246.2, -376.5)
2 (-252.9, -246.2, -375.25)
3 (-252.9, -246.2, -374.0)
4 (-252.9, -246.2, -372.75)


In [13]:
# 建立 CT ImagePositionPatient -> z-index 映射
ct_position_to_index = {}

for idx, f in enumerate(file_names):
    ds = pydicom.dcmread(f, stop_before_pixels=True)

    if hasattr(ds, "ImagePositionPatient"):
        pos = tuple(
            round(float(x), 4)
            for x in ds.ImagePositionPatient
        )

        ct_position_to_index[pos] = idx

# 根据空间位置重建3D mask
mask_array = np.zeros(
    (
        ct_image.GetSize()[2],
        ct_image.GetSize()[1],
        ct_image.GetSize()[0]
    ),
    dtype=np.uint8
)

matched_frames = 0

for frame_idx, pos in enumerate(frame_positions):

    if pos is None:
        continue

    pos_key = tuple(round(float(x), 4) for x in pos)

    if pos_key in ct_position_to_index:
        z_idx = ct_position_to_index[pos_key]

        mask_array[z_idx] = (
            seg_array[frame_idx] > 0
        ).astype(np.uint8)

        matched_frames += 1

print("SEG frames:", seg_array.shape[0])
print("Matched frames:", matched_frames)
print("Mask shape:", mask_array.shape)
print("Mask unique values:", np.unique(mask_array))
print("Mask nonzero voxels:", int(mask_array.sum()))

SEG frames: 288
Matched frames: 288
Mask shape: (288, 512, 512)
Mask unique values: [0 1]
Mask nonzero voxels: 1324


In [14]:
mask_image = sitk.GetImageFromArray(mask_array.astype(np.uint8))
mask_image.CopyInformation(ct_image)

print("CT size:", ct_image.GetSize())
print("Mask size:", mask_image.GetSize())

print("\nCT spacing:", ct_image.GetSpacing())
print("Mask spacing:", mask_image.GetSpacing())

print("\nCT origin:", ct_image.GetOrigin())
print("Mask origin:", mask_image.GetOrigin())

print("\nCT direction:", ct_image.GetDirection())
print("Mask direction:", mask_image.GetDirection())

CT size: (512, 512, 288)
Mask size: (512, 512, 288)

CT spacing: (0.9375, 0.9375, 1.25)
Mask spacing: (0.9375, 0.9375, 1.25)

CT origin: (-252.9, -246.2, -377.75)
Mask origin: (-252.9, -246.2, -377.75)

CT direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
Mask direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)


In [15]:
ct_arr = sitk.GetArrayFromImage(ct_image)
mask_arr = sitk.GetArrayFromImage(mask_image)

print("CT array shape:", ct_arr.shape)
print("Mask array shape:", mask_arr.shape)
print("Mask voxels:", int(mask_arr.sum()))

z_nonzero = np.where(mask_arr.sum(axis=(1, 2)) > 0)[0]

print("Tumor spans z-slices:")
print(z_nonzero[0], "to", z_nonzero[-1])
print("Number of slices with tumor:", len(z_nonzero))

CT array shape: (288, 512, 512)
Mask array shape: (288, 512, 512)
Mask voxels: 1324
Tumor spans z-slices:
138 to 147
Number of slices with tumor: 10


In [16]:
result = extractor.execute(ct_image, mask_image)

# 只保留真正的 radiomics 特征，排除 diagnostics
feature_result = {
    k: v
    for k, v in result.items()
    if not k.startswith("diagnostics_")
}

print("Number of extracted features:", len(feature_result))

for i, (k, v) in enumerate(feature_result.items()):
    print(k, "=", v)
    if i >= 14:
        break

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Number of extracted features: 107
original_shape_Elongation = 0.7202212574313953
original_shape_Flatness = 0.6571952088830638
original_shape_LeastAxisLength = 11.036269206551351
original_shape_MajorAxisLength = 16.79298488086674
original_shape_Maximum2DDiameterColumn = 15.264337522473747
original_shape_Maximum2DDiameterRow = 20.396078054371138
original_shape_Maximum2DDiameterSlice = 20.248456731316587
original_shape_Maximum3DDiameter = 21.908902300206645
original_shape_MeshVolume = 1436.0
original_shape_MinorAxisLength = 12.094664686924252
original_shape_Sphericity = 0.7417141109140835
original_shape_SurfaceArea = 829.8836524259049
original_shape_SurfaceVolumeRatio = 0.5779134069818279
original_shape_VoxelVolume = 1455.0
original_firstorder_10Percentile = -349.79999999999995


In [17]:
extract_case(case_id)

NameError: name 'extract_case' is not defined

In [18]:
extract_case(case_id)

NameError: name 'extract_case' is not defined

In [19]:
def extract_case(case_id, qc_df, extractor):
    """
    从 final_cohort_qc.csv 中读取指定患者的 CT 和 SEG，
    将 DICOM SEG 按 ImagePositionPatient 映射到 CT 空间，
    然后使用 PyRadiomics 提取 Original radiomics 特征。
    """

    # 1. 获取 CT 和 SEG 路径
    row = qc_df.loc[qc_df["Case ID"] == case_id].iloc[0]

    ct_folder = Path(row["Matched CT folder"])
    seg_file = Path(row["SEG file"])

    # 2. 读取 CT
    reader = sitk.ImageSeriesReader()

    series_ids = reader.GetGDCMSeriesIDs(str(ct_folder))

    if not series_ids:
        raise RuntimeError(f"{case_id}: No CT series found")

    file_names = reader.GetGDCMSeriesFileNames(
        str(ct_folder),
        series_ids[0]
    )

    reader.SetFileNames(file_names)
    ct_image = reader.Execute()

    # 3. 读取 SEG
    seg_ds = pydicom.dcmread(seg_file)
    seg_array = seg_ds.pixel_array

    # 4. 建立 CT 空间位置 -> z-index 映射
    ct_position_to_index = {}

    for idx, f in enumerate(file_names):
        ds = pydicom.dcmread(
            f,
            stop_before_pixels=True
        )

        if not hasattr(ds, "ImagePositionPatient"):
            continue

        pos = tuple(
            round(float(x), 4)
            for x in ds.ImagePositionPatient
        )

        ct_position_to_index[pos] = idx

    # 5. 获取每个 SEG frame 的空间位置
    frame_positions = []

    for frame_fg in seg_ds.PerFrameFunctionalGroupsSequence:
        position = None

        if hasattr(frame_fg, "PlanePositionSequence"):
            item = frame_fg.PlanePositionSequence[0]

            if hasattr(item, "ImagePositionPatient"):
                position = tuple(
                    float(x)
                    for x in item.ImagePositionPatient
                )

        frame_positions.append(position)

    # 6. 重建 3D mask
    mask_array = np.zeros(
        (
            ct_image.GetSize()[2],
            ct_image.GetSize()[1],
            ct_image.GetSize()[0]
        ),
        dtype=np.uint8
    )

    matched_frames = 0

    for frame_idx, pos in enumerate(frame_positions):

        if pos is None:
            continue

        pos_key = tuple(
            round(float(x), 4)
            for x in pos
        )

        if pos_key in ct_position_to_index:
            z_idx = ct_position_to_index[pos_key]

            mask_array[z_idx] = (
                seg_array[frame_idx] > 0
            ).astype(np.uint8)

            matched_frames += 1

    # 7. 基本 QC
    if matched_frames == 0:
        raise RuntimeError(
            f"{case_id}: No SEG frames matched CT"
        )

    if mask_array.sum() == 0:
        raise RuntimeError(
            f"{case_id}: Reconstructed mask is empty"
        )

    # 8. 转为 SimpleITK mask
    mask_image = sitk.GetImageFromArray(
        mask_array.astype(np.uint8)
    )

    mask_image.CopyInformation(ct_image)

    # 9. 提取 radiomics
    result = extractor.execute(
        ct_image,
        mask_image
    )

    # 10. 去掉 diagnostics，只保留真正特征
    features = {
        k: v
        for k, v in result.items()
        if not k.startswith("diagnostics_")
    }

    # 11. 检查特征数
    if len(features) != 107:
        raise RuntimeError(
            f"{case_id}: Expected 107 features, "
            f"but got {len(features)}"
        )

    return {
        "Case ID": case_id,
        "Matched frames": matched_frames,
        "Mask voxels": int(mask_array.sum()),
        **features
    }

In [20]:
test_result = extract_case(
    "R01-003",
    qc_df,
    extractor
)

print("Case:", test_result["Case ID"])
print("Matched frames:", test_result["Matched frames"])
print("Mask voxels:", test_result["Mask voxels"])

feature_names = [
    k for k in test_result
    if k.startswith("original_")
]

print("Radiomics features:", len(feature_names))

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Case: R01-003
Matched frames: 288
Mask voxels: 1324
Radiomics features: 107


In [21]:
test_cases = [
    "R01-003",
    "R01-005",
    "R01-023",
    "R01-133",
    "R01-150"
]

test_results = []

for case_id in test_cases:
    try:
        result = extract_case(
            case_id,
            qc_df,
            extractor
        )

        test_results.append({
            "Case ID": case_id,
            "Status": "PASS",
            "Matched frames": result["Matched frames"],
            "Mask voxels": result["Mask voxels"],
            "Feature count": len([
                k for k in result
                if k.startswith("original_")
            ])
        })

        print(
            case_id,
            "PASS",
            "| matched:", result["Matched frames"],
            "| voxels:", result["Mask voxels"]
        )

    except Exception as e:
        test_results.append({
            "Case ID": case_id,
            "Status": "FAIL",
            "Error": str(e)
        })

        print(case_id, "FAIL:", e)

test_df = pd.DataFrame(test_results)
display(test_df)

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


R01-003 PASS | matched: 288 | voxels: 1324


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


R01-005 PASS | matched: 329 | voxels: 16625


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


R01-023 PASS | matched: 69 | voxels: 133455


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


R01-133 PASS | matched: 39 | voxels: 54547
R01-150 FAIL: single positional indexer is out-of-bounds


,Case ID,Status,Matched frames,Mask voxels,Feature count,Error
0,R01-003,PASS,288.0,1324.0,107.0,NaN
1,R01-005,PASS,329.0,16625.0,107.0,NaN
2,R01-023,PASS,69.0,133455.0,107.0,NaN
3,R01-133,PASS,39.0,54547.0,107.0,NaN
4,R01-150,FAIL,NaN,NaN,NaN,single positional indexer is out-of-bounds


In [22]:
print("R01-150 in final cohort:",
      "R01-150" in set(qc_df["Case ID"].astype(str).str.strip()))

print("Number of final cohort patients:", len(qc_df))

R01-150 in final cohort: False
Number of final cohort patients: 97


In [23]:
tested_cases = {
    "R01-003",
    "R01-005",
    "R01-023",
    "R01-133"
}

valid_ids = qc_df["Case ID"].astype(str).str.strip().tolist()

replacement_case = next(
    cid for cid in valid_ids
    if cid not in tested_cases
)

print("Replacement test case:", replacement_case)

Replacement test case: R01-006


In [24]:
result = extract_case(
    replacement_case,
    qc_df,
    extractor
)

print("Case:", result["Case ID"])
print("Matched frames:", result["Matched frames"])
print("Mask voxels:", result["Mask voxels"])

feature_count = sum(
    k.startswith("original_")
    for k in result
)

print("Radiomics features:", feature_count)

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Case: R01-006
Matched frames: 567
Mask voxels: 7050
Radiomics features: 107


In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import traceback

In [26]:
# 输出目录
FEATURE_OUTPUT = PROJECT_ROOT / "data" / "processed" / "radiomics_features.csv"
LOG_OUTPUT = PROJECT_ROOT / "results" / "metrics" / "radiomics_extraction_log.csv"

FEATURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

all_features = []
extraction_log = []

case_ids = (
    qc_df["Case ID"]
    .astype(str)
    .str.strip()
    .tolist()
)

print("Total cases to process:", len(case_ids))

Total cases to process: 97


In [27]:
for i, case_id in enumerate(case_ids, start=1):

    print(f"[{i:03d}/{len(case_ids)}] {case_id} ...", end=" ")

    try:
        result = extract_case(
            case_id,
            qc_df,
            extractor
        )

        # 提取真正的107个影像组学特征
        feature_names = [
            k for k in result.keys()
            if k.startswith("original_")
        ]

        feature_row = {
            "Case ID": case_id,
            **{
                k: float(result[k])
                for k in feature_names
            }
        }

        all_features.append(feature_row)

        extraction_log.append({
            "Case ID": case_id,
            "Status": "PASS",
            "Matched frames": result["Matched frames"],
            "Mask voxels": result["Mask voxels"],
            "Feature count": len(feature_names),
            "Error": None
        })

        print(
            f"PASS | frames={result['Matched frames']} "
            f"| voxels={result['Mask voxels']} "
            f"| features={len(feature_names)}"
        )

    except Exception as e:

        extraction_log.append({
            "Case ID": case_id,
            "Status": "FAIL",
            "Matched frames": None,
            "Mask voxels": None,
            "Feature count": None,
            "Error": str(e)
        })

        print("FAIL:", e)

[001/97] R01-003 ... 

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


PASS | frames=288 | voxels=1324 | features=107
[002/97] R01-005 ... 

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


PASS | frames=329 | voxels=16625 | features=107
[003/97] R01-006 ... 

...

PASS | frames=12 | voxels=7039 | features=107
[096/97] R01-145 ... 

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


PASS | frames=38 | voxels=60195 | features=107
[097/97] R01-146 ... 

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


PASS | frames=112 | voxels=240696 | features=107


In [29]:
print("Feature table shape:", features_df.shape)

# 只取107个特征列
feature_cols = [
    c for c in features_df.columns
    if c.startswith("original_")
]

print("Radiomics feature count:", len(feature_cols))

# 缺失值
missing_count = features_df[feature_cols].isna().sum().sum()

# Inf
inf_count = np.isinf(
    features_df[feature_cols].astype(float).values
).sum()

# 零方差特征
zero_var_features = [
    c for c in feature_cols
    if features_df[c].nunique(dropna=False) <= 1
]

print("Total missing values:", missing_count)
print("Total Inf values:", inf_count)
print("Zero-variance features:", len(zero_var_features))

if zero_var_features:
    print(zero_var_features)

NameError: name 'features_df' is not defined

In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"F:\yingxiang2")

FEATURE_OUTPUT = PROJECT_ROOT / "data" / "processed" / "radiomics_features.csv"
LOG_OUTPUT = PROJECT_ROOT / "results" / "metrics" / "radiomics_extraction_log.csv"

features_df = pd.read_csv(FEATURE_OUTPUT)
log_df = pd.read_csv(LOG_OUTPUT)

print("Feature table loaded:", features_df.shape)
print("Log table loaded:", log_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'F:\\yingxiang2\\data\\processed\\radiomics_features.csv'

In [31]:
print("all_features exists:", "all_features" in globals())
print("extraction_log exists:", "extraction_log" in globals())

all_features exists: True
extraction_log exists: True


In [32]:
features_df = pd.DataFrame(all_features)
log_df = pd.DataFrame(extraction_log)

FEATURE_OUTPUT = PROJECT_ROOT / "data" / "processed" / "radiomics_features.csv"
LOG_OUTPUT = PROJECT_ROOT / "results" / "metrics" / "radiomics_extraction_log.csv"

FEATURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

features_df.to_csv(FEATURE_OUTPUT, index=False)
log_df.to_csv(LOG_OUTPUT, index=False)

print("Feature table:", features_df.shape)
print("Saved:", FEATURE_OUTPUT.exists(), FEATURE_OUTPUT)

print("Log table:", log_df.shape)
print("Saved:", LOG_OUTPUT.exists(), LOG_OUTPUT)

Feature table: (97, 108)
Saved: True F:\yingxiang2\data\processed\radiomics_features.csv
Log table: (97, 6)
Saved: True F:\yingxiang2\results\metrics\radiomics_extraction_log.csv


In [33]:
feature_cols = [
    c for c in features_df.columns
    if c.startswith("original_")
]

missing_count = features_df[feature_cols].isna().sum().sum()

inf_count = np.isinf(
    features_df[feature_cols].astype(float).values
).sum()

zero_var_features = [
    c for c in feature_cols
    if features_df[c].nunique(dropna=False) <= 1
]

print("Feature table shape:", features_df.shape)
print("Radiomics feature count:", len(feature_cols))
print("Total missing values:", missing_count)
print("Total Inf values:", inf_count)
print("Zero-variance features:", len(zero_var_features))

if zero_var_features:
    print(zero_var_features)

print("\nExtraction status:")
print(log_df["Status"].value_counts(dropna=False))

print("\nFeature count distribution:")
print(log_df["Feature count"].value_counts(dropna=False))

Feature table shape: (97, 108)
Radiomics feature count: 107
Total missing values: 0
Total Inf values: 0
Zero-variance features: 0

Extraction status:
Status
PASS    97
Name: count, dtype: int64

Feature count distribution:
Feature count
107    97
Name: count, dtype: int64
